# Organizando experimentos

## 1 Convenções

Quando trabalhamos em equipe e/ou a medida que os projetos crescem, se torna necessária a adoção de convenções para organização do histórico de projetos no MLFlow. Dessa forma, o histórico será de fato um registro documental que permite a rastreabilidade do que foi avaliado e como o desenvolvimento aconteceu. Os principais itens que compõem uma boa _checklist_ de rastreabilidade são:

1. **Objetivos da execução:** Deve ficar claro o que se pretendia descobrir ou validar, qual era a hipótese testada.
2. **Origem dos dados:** Registrar qual base de dados foi utilizada (versão do dataset).
3. **Versão do código:** Associar a execução a um commit do Git, permitindo saber exatamente qual código gerou aquele resultado.
4. **Pré-processamento:** Quais filtros ou técnicas de transformação foram aplicados aos dados.
5. **Estratégia de avaliação:** Documentar como foi feita a divisão dos dados (estratégia de split).
6. **Resultados:** As métricas obtidas e os artefatos gerados (gráficos, relatórios) que auxiliam na interpretação.
7. **Decisão:** Registrar qual foi a decisão tomada após a análise daquela execução (por exemplo, se o modelo foi descartado, se é um candidato ou se tornou o campeão).

### 1.1 Experimentos

O experimentos agrupam _runs_ por projeto, fase e objetivo. A recomendação é utilizar uma estrutura que responda rapidamente a três perguntas: _**"qual o projeto"**_, _**"qual a fase"**_ e _**"qual o objetivo"**_ daquela bateria de testes.

O padrão pode ser algo como:

> `<projeto>_<fase>_<objetivo>`

O importante é que, ao olhar para a lista, qualquer pessoa da equipe saiba identificar o contexto daquela série de experimentos.

### 1.2 _Runs_

Para as execuções, a convenção deve facilitar a leitura e a identificação do histórico. Uma prática recomendada é descrver: _**"a etapa do projeto"**_, _**"o modelo utilizado"**_, _**"o objetivo específico daquela rodada"**_ e _**"a versão daquela execução"**_. Algo como: aula, modelo, objetivo e versão. Isso cria uma trilha cronológica e lógica do desenvolvimento.

> `<etapa>_<modelo>_<objetivo>_<versao>`

### 1.3 _Tags_

Descrevem o contexto e permitem filtragens. A convenção aqui é separar as _tags_ por grupos de responsabilidade:

- **Projeto:** identifica a qual iniciativa a run pertence. Ex: `project`, `task`, `target`;
- **Dados:** armazena a versão do dataset utilizado. Ex: `dataset_version`, `feature_set`, `split_strategy`;
- **Código:** registra a versão do commit para garantir a reprodutibilidade. Ex: `code_version`, `git_commit`;
- **Experimento:** descreve a hipótese que está sendo testada. Ex: `hypothesis`, `model_name`, `model_family`, `tracking_type`, `run_porpuse`;
  - Como exemplo ruim de hipótese podemos citar: _“testar um random forest”_, que é um texto vago e não informa que critério foi usado.
  - Um texto mais adequado para o que buscamos é: _"avaliar se um modelo Random Forest melhora a métrica ROC-AUC e recall (sensibilidade) da classe inadimplente em relação ao baseline de Regressão Logística, aceitando menor interpretabilidade se o ganho técnico for relevante"_. Isso ajuda muito quando precisamos justificar por que executamos a _run_.
- **Decisão:** indica o status daquela execução, como candidato, desafiante ou campeão. Ex: `selection_status`, `selected_for_registry`, `selection_rule`.

Uma dica valiosa é centralizar essas _tags_ em um dicionário comum, que pode ser reutilizado, evitando repetição de código e erros de digitação que possam criar categorias duplicadas ou confusas.

In [1]:
# capturando commit hash do git para rastreabilidade do codigo
import subprocess

def get_git_commit(default="not_available"):
    try:
        commit = subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"],
            text=True,
        ).strip()
        return commit
    except Exception:
        return default


CODE_VERSION = get_git_commit()
CODE_VERSION

'230a791'

In [2]:
# define tags comuns para todas as runs, para melhor rastreabilidade do experimento
COMMON_TAGS = {
    "project":         "risco_credito",
    "task":            "binary_classification",
    "target":          "inadimplente",
    "dataset_version": "synthetic_credit_v1",
    "feature_set":     "core_features_v1",
    "split_strategy":  "train_test_stratified_75_25",
    "code_version":    CODE_VERSION,
}
COMMON_TAGS

{'project': 'risco_credito',
 'task': 'binary_classification',
 'target': 'inadimplente',
 'dataset_version': 'synthetic_credit_v1',
 'feature_set': 'core_features_v1',
 'split_strategy': 'train_test_stratified_75_25',
 'code_version': '230a791'}

### 1.4 Artefatos

Permitem o armazenamento de evidências para auditoria e colaboração. A convenção é organizar os arquivos de saída em pastas temáticas dentro da estrutura da execução. As pastas sugeridas são: 

```text
artifacts/
├── data/                                   # resumo das bases e versões dos dados processados
│   └── dataset_profile.json   
├── reports/                                # relatórios de desempenho em formato de texto ou markdown
│   ├── classification_report.txt
│   ├── run_summary.md
│   └── champion_summary.md
├── plots/                                  # gráficos e visualizações de avaliação
│   ├── matriz_confusao.png
│   ├── curva_roc.png
│   └── coeficientes_modelo.png
├── tables/                                 # tabelas reutilizáveis
│   └── coeficientes_modelo.json
└── selection                               # documentos ou registros que sustentam a decisão sobre a escolha do modelo campeão
    ├── decision_matrix.json
    └── registry_preparations.json
```

## 2 Elegendo o modelo campeão

O próximo passo é eleger um modelo campeão, o objetivo é selecionar o modelo mais adequado ao contexto atual, baseando essa escolha em critérios bem definidos.

Para isso devemos comparar: desempenho, interpretabilidade e custo operacional. Não devemos apenas escolher o modelo que possui o maior valor em uma métrica específica, mas sim decidir de acordo com critérios objetivos que equilibrem desempenho e operação, formando assim uma decisão embasada.

Mas primeiro, vamos levantar o servidor local do MLflow para que possamos acessar a _user interface_ . Para isso rodamos o comando abaixo no terminal

```terminal
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

e depois acessamos o endereço [http://127.0.0.1:5000](http://127.0.0.1:5000).

### 2.1 Selecionando as _runs_ de comparação

Dentre os modelos candidatos temos:

- **Regressão logística:** tende a ser mais simples, rápida e interpretável
- **Árvore de decisão:** também é relativamente simples, porém pode ser instável
- **Random Forest:** tende a ser mais robusto, mas é menos direto de explicar e pode ter custo operacional elevado.

Vamos começar estabelecendo o _tracking_ com o banco de dados e selecionar as _runs_ de comparação:

In [3]:
import mlflow

# define a ligacao com o banco de dados relacional ja existente, onde o MLflow vai armazenar os metadados do experimento
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# recupera experimento pelo nome
EXPERIMENT_NAME = "risco_credito_comparacao_modelos"
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# filtra runs do experimento de comparacao usando tags
df_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_purpose = 'compare_model_candidates'",
    order_by=["metrics.roc_auc_teste DESC"],
)

if df_runs.empty:
    raise ValueError("Nenhuma run para comparação foi encontrada para seleção.")

df_runs.shape

(3, 115)

Como o DataFrame das _runs_ de comparação possui mais de 100 colunas, precisamos reduzi-lo para conter apenas as colunas relevantes para a comparação, mantendo as métricas principais (ROC, _precision_, _recall_ e F1), o nome do modelo e identificação da _run_. E além disso, os nomes das colunas podem ser normalizados:

In [4]:
model_col     = ("params.model_name" if "params.model_name" in df_runs.columns else "tags.model_name") # determina coluna com nome do modelo
decision_cols = [
    "run_id",
    "tags.mlflow.runName",
    model_col,
    "metrics.roc_auc_teste",
    "metrics.roc_auc_cv_media",
    "metrics.roc_auc_cv_desvio",
    "metrics.accuracy_teste",
    "metrics.precision_inadimplente",
    "metrics.recall_inadimplente",
    "metrics.f1_inadimplente",
]

available_cols = [col for col in decision_cols if col in df_runs.columns] # garante que apenas colunas existentes no dataframe sejam selecionadas
df_decision    = df_runs[available_cols].copy()
df_decision    = df_decision.rename(
    columns={
        "tags.mlflow.runName": "run_name",
        model_col: "model_name",
        "metrics.roc_auc_teste": "roc_auc_teste",
        "metrics.roc_auc_cv_media": "roc_auc_cv_media",
        "metrics.roc_auc_cv_desvio": "roc_auc_cv_desvio",
        "metrics.accuracy_teste": "accuracy_teste",
        "metrics.precision_inadimplente": "precision_inadimplente",
        "metrics.recall_inadimplente": "recall_inadimplente",
        "metrics.f1_inadimplente": "f1_inadimplente",
    }
)

metric_cols = [
    "roc_auc_teste",
    "roc_auc_cv_media",
    "roc_auc_cv_desvio",
    "accuracy_teste",
    "precision_inadimplente",
    "recall_inadimplente",
    "f1_inadimplente",
]

df_decision[metric_cols] = df_decision[metric_cols].astype(float)
df_decision              = df_decision.sort_values("roc_auc_teste", ascending=False)

print(df_decision.shape)
df_decision

(3, 10)


,run_id,run_name,model_name,roc_auc_teste,roc_auc_cv_media,roc_auc_cv_desvio,accuracy_teste,precision_inadimplente,recall_inadimplente,f1_inadimplente
0,e4afeca8be834b3fb69b1dbd6e3f4c0f,hibrid_log_comparison_logistic_regression,logistic_regression,0.903577,0.902266,0.001095,0.822667,0.389034,0.823204,0.528369
1,9ca31560460247d38490485103e1a03f,hibrid_log_comparison_random_forest,random_forest,0.894131,0.889951,0.001628,0.803333,0.364929,0.850829,0.510779
2,7a7bb50e45434c29a3840190f02460f4,hibrid_log_comparison_decision_tree,decision_tree,0.890569,0.875530,0.004451,0.800667,0.362791,0.861878,0.510638


### 2.2 Política de seleção

Como a decisão será tomada de acordo com múltiplos critérios, faz sentido definir uma **política de seleção** em que são atribuídos pesos para cada um dos critérios (técnicos e operacionais) que reflitam sua importância relativa ao contexto do projeto. Dessa forma, atribuindo valores numéricos para cada critério, poderemos calcular um _score_ composto combinando fatores técnicos e operacionais.

Esse método permite que a decisão não se baseie apenas no desempenho absoluto de alguma métrica, mas sim no equilíbrio entre desempenho, interpretabilidade e simplicidade operacional. Lembrando que a definição dos pesos deve considerar as necessidades do negócio, pois diferentes contextos podem valorizar mais a interpretabilidade do que desempenho puro em alguma métrica técnica.

In [5]:
# define pesos para cada criterio de selecao do modelo campeao
selection_policy = {
    "technical_score": {
        "roc_auc_teste": 0.35,
        "recall_inadimplente": 0.25,
        "precision_inadimplente": 0.15,
        "f1_inadimplente": 0.15,
        "stability_score": 0.10, # vai depender da validacao cruzada (cv), desvio padrao do roc_auc
    },
    "final_score": {
        "technical_score": 0.75,
        "interpretability_score": 0.15,
        "operational_simplicity_score": 0.10, # custo computacional, facilidade de deploy, facilidade de monitoramento, etc.
    },
    "notes": (
        "Pesos adotados para seleção do modelo campeão nesse projeto. "
        "Em um projeto real, esses pesos devem ser definidos "
        "considerando negócio, risco, compliance e engenharia."
    ),
}

selection_policy

{'technical_score': {'roc_auc_teste': 0.35,
  'recall_inadimplente': 0.25,
  'precision_inadimplente': 0.15,
  'f1_inadimplente': 0.15,
  'stability_score': 0.1},
 'final_score': {'technical_score': 0.75,
  'interpretability_score': 0.15,
  'operational_simplicity_score': 0.1},
 'notes': 'Pesos adotados para seleção do modelo campeão nesse projeto. Em um projeto real, esses pesos devem ser definidos considerando negócio, risco, compliance e engenharia.'}

In [6]:
# atribui scores de interpretabilidade e simplicidade operacional para cada modelo
interpretability_score                      = {"logistic_regression": 1.00, "decision_tree": 0.80, "random_forest": 0.50}
operational_simplicity_score                = {"logistic_regression": 1.00, "decision_tree": 0.90, "random_forest": 0.65}

df_decision["interpretability_score"]       = (df_decision["model_name"].map(interpretability_score).fillna(0.50))
df_decision["operational_simplicity_score"] = (df_decision["model_name"].map(operational_simplicity_score).fillna(0.50))
df_decision["stability_score"]              = (1 - df_decision["roc_auc_cv_desvio"].clip(lower=0, upper=1)) # quanto menor o desvio padrao do roc_auc na validacao cruzada, maior a estabilidade do modelo

df_decision

,run_id,run_name,model_name,roc_auc_teste,roc_auc_cv_media,roc_auc_cv_desvio,accuracy_teste,precision_inadimplente,recall_inadimplente,f1_inadimplente,interpretability_score,operational_simplicity_score,stability_score
0,e4afeca8be834b3fb69b1dbd6e3f4c0f,hibrid_log_comparison_logistic_regression,logistic_regression,0.903577,0.902266,0.001095,0.822667,0.389034,0.823204,0.528369,1.0,1.00,0.998905
1,9ca31560460247d38490485103e1a03f,hibrid_log_comparison_random_forest,random_forest,0.894131,0.889951,0.001628,0.803333,0.364929,0.850829,0.510779,0.5,0.65,0.998372
2,7a7bb50e45434c29a3840190f02460f4,hibrid_log_comparison_decision_tree,decision_tree,0.890569,0.875530,0.004451,0.800667,0.362791,0.861878,0.510638,0.8,0.90,0.995549


### 2.3 Calculando _scores_ e selecionando o modelo candidato a campeão

E finalmente calculamos os _scores_ e determinamos o modelo campeão:

In [7]:
# calcula score tecnico e score final para cada modelo candidato, considerando pesos definidos na politica de selecao
df_decision["technical_score"] = df_decision.apply(lambda row: sum([v * row[k] for k, v in selection_policy["technical_score"].items()]), axis=1)
df_decision["final_score"]     = df_decision.apply(lambda row: sum([v * row[k] for k, v in selection_policy["final_score"].items()]), axis=1)

# gera ranking final dos modelos candidatos, considerando score final e desempenho no conjunto de teste
df_decision_ranking = (
    df_decision
    .sort_values(["final_score", "roc_auc_teste"], ascending=False)
    .reset_index(drop=True)
)

df_decision_ranking

,run_id,run_name,model_name,roc_auc_teste,roc_auc_cv_media,roc_auc_cv_desvio,accuracy_teste,precision_inadimplente,recall_inadimplente,f1_inadimplente,interpretability_score,operational_simplicity_score,stability_score,technical_score,final_score
0,e4afeca8be834b3fb69b1dbd6e3f4c0f,hibrid_log_comparison_logistic_regression,logistic_regression,0.903577,0.902266,0.001095,0.822667,0.389034,0.823204,0.528369,1.0,1.00,0.998905,0.759554,0.819665
1,7a7bb50e45434c29a3840190f02460f4,hibrid_log_comparison_decision_tree,decision_tree,0.890569,0.875530,0.004451,0.800667,0.362791,0.861878,0.510638,0.8,0.90,0.995549,0.757738,0.778303
2,9ca31560460247d38490485103e1a03f,hibrid_log_comparison_random_forest,random_forest,0.894131,0.889951,0.001628,0.803333,0.364929,0.850829,0.510779,0.5,0.65,0.998372,0.756847,0.707635


In [8]:
# define o modelo campeao como o primeiro colocado no ranking final, considerando score final e desempenho no conjunto de teste
df_champion = df_decision_ranking.iloc[0]
champion    = df_champion.to_dict()

df_champion[['model_name', 'run_name', 'run_id']]

model_name                          logistic_regression
run_name      hibrid_log_comparison_logistic_regression
run_id                 e4afeca8be834b3fb69b1dbd6e3f4c0f
Name: 0, dtype: object

### 2.4 Documentando o modelo campeão

Escrevemos o resumo reunindo informações necessárias para justificar a escolha do modelo candidato a campeão, temos: o contexto, o modelo selecionado, as métricas principais, o critério de seleção (pesos técnicos e o _score_ final) e a justificativa.

Outras informações que também costumam ser incluídas no resumo da decisão são: as limitações, os riscos, quais dados foram utilizados, métricas por segmento, análise de falhas e critérios de monitoramento.

In [9]:
champion_summary = (
    f"# Seleção do modelo campeão\n"
    f"\n"
    f"## Contexto\n"
    f"\n"
    f"Projeto: risco de crédito \n"
    f"Tarefa: classificação binária \n"
    f"Target: inadimplente \n"
    f"Dataset version: synthetic_credit_v1 \n"
    f"Feature set: core_features_v1 \n"
    f"\n"
    f"## Modelo selecionado\n"
    f"\n"
    f"- Modelo: {champion['model_name']}\n"
    f"- Run name: {champion['run_name']}\n"
    f"- Run ID: {champion['run_id']}\n"
    f"\n"
    f"## Principais métricas\n"
    f"\n"
    f"- ROC-AUC teste: {champion["roc_auc_teste"]:.4f}\n"
    f"- ROC-AUC CV média: {champion["roc_auc_cv_media"]:.4f}\n"
    f"- ROC-AUC CV desvio: {champion["roc_auc_cv_desvio"]:.4f}\n"
    f"- Precision inadimplente: {champion["precision_inadimplente"]:.4f}\n"
    f"- Recall inadimplente: {champion["recall_inadimplente"]:.4f}\n"
    f"- F1 inadimplente: {champion["f1_inadimplente"]:.4f}\n"
    f"\n"
    f"## Critérios de decisão\n"
    f"\n"
    f"A seleção considerou desempenho técnico, estabilidade, interpretabilidade e simplicidade operacional.\n"
    f"\n"
    f"Pesos técnicos:\n"
    f"- ROC-AUC teste: 35%\n"
    f"- Recall inadimplente: 25%\n"
    f"- Precision inadimplente: 15%\n"
    f"- F1 inadimplente: 15%\n"
    f"- Estabilidade: 10%\n"
    f"\n"
    f"Score final:\n"
    f"- Score técnico: 75%\n"
    f"- Interpretabilidade: 15%\n"
    f"- Simplicidade operacional: 10%\n"
    f"\n"
    f"## Justificativa\n"
    f"\n"
    f"O modelo selecionado apresentou o melhor equilíbrio entre desempenho e critérios operacionais dentro da política de seleção definida.\n"
    f"\n"
    f"## Próximo passo\n"
    f"\n"
    f"Preparar este modelo para registro no MLflow Model Registry.\n"
)

print(champion_summary)

# Seleção do modelo campeão

## Contexto

Projeto: risco de crédito 
Tarefa: classificação binária 
Target: inadimplente 
Dataset version: synthetic_credit_v1 
Feature set: core_features_v1 

## Modelo selecionado

- Modelo: logistic_regression
- Run name: hibrid_log_comparison_logistic_regression
- Run ID: e4afeca8be834b3fb69b1dbd6e3f4c0f

## Principais métricas

- ROC-AUC teste: 0.9036
- ROC-AUC CV média: 0.9023
- ROC-AUC CV desvio: 0.0011
- Precision inadimplente: 0.3890
- Recall inadimplente: 0.8232
- F1 inadimplente: 0.5284

## Critérios de decisão

A seleção considerou desempenho técnico, estabilidade, interpretabilidade e simplicidade operacional.

Pesos técnicos:
- ROC-AUC teste: 35%
- Recall inadimplente: 25%
- Precision inadimplente: 15%
- F1 inadimplente: 15%
- Estabilidade: 10%

Score final:
- Score técnico: 75%
- Interpretabilidade: 15%
- Simplicidade operacional: 10%

## Justificativa

O modelo selecionado apresentou o melhor equilíbrio entre desempenho e critérios operac

E finalmente podemos registrar a decisão diretamente na _run_ vencedora, adicionando _tags_ e artefatos. Instânciando um _client_ com [`MlflowClient`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.client.html) para sermos capazes de adicionar novas _tags_ nessa run específica e também salvar artefatos com a decisão, a política de seleção e a matriz final:

In [ ]:
MODEL_NAME = "credit_risk_classifier"
client     = mlflow.MlflowClient()

client.set_tag(champion['run_id'], "selection_status",      "champion_candidate")                   # indica que o modelo foi selecionado como candidato a campeao
client.set_tag(champion['run_id'], "registry_model_name",   MODEL_NAME)                             # indica nome do modelo
client.set_tag(champion['run_id'], "selection_rule",        "weighted_score_v1")                    # indica regra de selecao usada para escolher o modelo campeao
client.set_tag(champion['run_id'], "selected_for_registry", "true")                                 # indica que o modelo foi selecionado para registro

mlflow.log_text(champion_summary,     "selection/champion_summary.md", run_id=champion['run_id'])   # explica porque o modelo foi selecionado como campeao
mlflow.log_dict(selection_policy,     "selection/selection_policy.json", run_id=champion['run_id']) # registra politica que levou a escolha do modelo campeao
mlflow.log_table(df_decision_ranking, "selection/decision_matrix.json", run_id=champion['run_id'])  # registra matriz de decisao dos modelos candidatos

Além dos artefatos acima, adicionamos também um artefato para a "preparação do registro", que é um exercício de organização e governança. O **Model Registry** é o local onde os modelos são promovidos para serem usados em produção. Não devemos simplesmente "jogar" um modelo lá, é preciso garantir que seja possível determinar exatamente de onde o modelo veio, por que ele foi escolhido e quais critérios foram usados para validá-lo.

In [ ]:
MODEL_ARTIFACT_PATH  = "model"

# path dentro da run onde o artefato do modelo esta salvo, local que o MLflow usara para move-lo para o registro
champion_model_uri   = f"runs:/{champion['run_id']}/{MODEL_ARTIFACT_PATH}"

# prepara metadados para registro do modelo campeao no MLflow Model Registry
registry_preparation = {
    "registered_model_name":      MODEL_NAME,
    "source_run_id":              champion['run_id'],
    "source_run_name":            champion['run_name'],
    "model_name":                 champion['model_name'],
    "model_uri":                  champion_model_uri,
    "intended_stage_next_lesson": "Staging",
    "selection_status":           "champion_candidate",
    "notes":                      "Metadados preparados registro efetivo no Model Registry."
}

# registra os metadados na run do modelo campeao no MLflow
mlflow.log_dict(registry_preparation, "selection/registry_preparation.json", run_id=champion['run_id'])

registry_preparation

{'registered_model_name': 'credit_risk_classifier',
 'source_run_id': 'e4afeca8be834b3fb69b1dbd6e3f4c0f',
 'source_run_name': 'hibrid_log_comparison_logistic_regression',
 'model_name': 'logistic_regression',
 'model_uri': 'runs:/e4afeca8be834b3fb69b1dbd6e3f4c0f/model',
 'intended_stage_next_lesson': 'Staging',
 'selection_status': 'champion_candidate',
 'notes': 'Metadados preparados registro efetivo no Model Registry.'}

Nessa preparação, definimos o nome para o modelo registrado (`"credit_risk_classifier"`) e a variável `champion_model_uri` que aponta para o artefato do modelo dentro da _run_ vencedora (a URI assume que ele foi salvo na pasta `model` pelo autolog).